In [1]:
from astroquery.gaia import Gaia

query = """
SELECT TOP 50
    g.source_id AS SOURCE_ID,
    g.ra,
    g.dec
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.nss_two_body_orbit AS n
ON g.source_id = n.source_id
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
results.write('gaia_binaries.csv', format='csv', overwrite=True)
print("Available columns:", results.colnames)
print(results[['SOURCE_ID', 'ra', 'dec']])

INFO: Query finished. [astroquery.utils.tap.core]
Available columns: ['SOURCE_ID', 'ra', 'dec']
     SOURCE_ID              ra                 dec         
                           deg                 deg         
------------------- ------------------ --------------------
5706094984537873792 126.55116979755597  -21.468906634210562
5706213937953041152 127.46800270351164   -21.05648890009294
5706273453317449472 126.87056104102183   -20.94620501105039
5706437619848657664 125.36428522928686  -20.662512579795425
5706530292361510272 127.11834569760813   -20.12976426934767
5706720812813052416 127.59291062329936  -20.063011932820114
5707359976964595200 125.79823618962087  -19.589903238558072
5706903439119704192  128.6532800627275  -18.939413392598496
5707008335104427648  126.8018871264103  -19.265272971185873
5707023109790382336 127.47485819389708   -19.14768732842094
5707824585050001536 126.14150101846678   -19.02175850803876
5941936071175766912  245.1720276725215  -46.658853822255146
    

In [2]:
import os
os.getcwd()

'C:\\Users\\holnc\\anaconda_projects\\a3360946-2a60-4ff6-b498-c7b0a22a9c0d'

In [3]:
from astroquery.gaia import Gaia

query = """
SELECT TOP 50
    g.source_id AS SOURCE_ID,
    g.ra,
    g.dec
FROM gaiadr3.gaia_source AS g
JOIN external.gaiaedr3_gcns_main_1 AS n
ON g.source_id = n.source_id
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
results.write('gaia_allstar.csv', format='csv', overwrite=True)
print('Max RA:', results['ra'].max())
print('Max DEC:', results['dec'].max())
print('Min RA:', results['ra'].min())
print('Min DEC:', results['dec'].min())
print("Available columns:", results.colnames)
print(results[['SOURCE_ID', 'ra', 'dec']])

INFO: Query finished. [astroquery.utils.tap.core]
Max RA: 335.1528898741702
Max DEC: 66.66104653162883
Min RA: 54.807872190015374
Min DEC: -57.57678559702253
Available columns: ['SOURCE_ID', 'ra', 'dec']
     SOURCE_ID              ra                 dec        
                           deg                 deg        
------------------- ------------------ -------------------
1872046609345556480  316.7484792940004   38.76386244649797
1872313550151474048 316.24883018126246  39.438381954802146
2165305464489588096  317.4888946386376   48.19197520792672
2164880327134774784 319.28578342914096  48.143643051432925
1958639369133397632  335.1528898741702   43.14576374558416
2078271800515663616  294.5977572734759    43.5140316644263
2079455699663132160 298.27080607369663   45.86012039093017
2085469199329073280  298.7052728184903   46.17558793547915
2085589355339132288 299.44844428356754   46.66043398905666
5934400606944882944  246.9879899211378  -51.08502219688215
5932610533368689664 242.09124

In [4]:
from astropy.table import Table
import pandas as pd

df = pd.read_csv("Source_ID_ONLY - Initial Cuts.csv")
df = df.iloc[:, 0:1]
df.columns = ["source_id"]
df["source_id"] = df["source_id"].astype("int64")

t = Table.from_pandas(df)

t.write("init_cut_clean.vot", format="votable", overwrite=True)

query = """
SELECT TOP 100
    g.source_id,
    g.ra,
    g.dec,
    g.parallax,
    g.phot_g_mean_mag,
    g.phot_bp_mean_mag,
    g.phot_rp_mean_mag,
    g.phot_bp_mean_mag - g.phot_rp_mean_mag AS bp_rp

FROM gaiadr3.gaia_source AS g
JOIN TAP_UPLOAD.initial_cuts AS u
ON g.source_id = u.source_id

WHERE g.phot_g_mean_mag IS NOT NULL
  AND g.phot_bp_mean_mag IS NOT NULL
  AND g.phot_rp_mean_mag IS NOT NULL
  AND g.parallax IS NOT NULL

ORDER BY u.source_id
"""

job = Gaia.launch_job(   # <-- use sync for now (more stable)
    query=query,
    upload_resource="init_cut_clean.vot",
    upload_table_name="initial_cuts"
)

results = job.get_results()

results.write("gaia_gbprp_data.vot", format="csv", overwrite=True)

print(results[['SOURCE_ID','ra','dec','parallax','phot_g_mean_mag','phot_bp_mean_mag','phot_rp_mean_mag', 'bp_rp']])
print(results.colnames)

HTTPError: Error 500: 
Can't overwrite cause with esavo.tap.upload.exceptions.UploadUwsException: DB quota exceeded for user anonymous (Currently 1000 MB, maximum allowed is 1000 MB). Registered users have an increased quota, please log in.


In [ ]:
from astropy.table import Table
import pandas as pd

df = pd.read_csv("Source_ID_ONLY - Pot Bins.csv")
df = df.iloc[:, 0:1]
df.columns = ["source_id"]
df["source_id"] = df["source_id"].astype("int64")

t = Table.from_pandas(df)

t.write("ruwe_cut_clean.vot", format="votable", overwrite=True)

query = """
SELECT TOP 100
    g.source_id,
    g.ra,
    g.dec,
    g.parallax,
    g.phot_g_mean_mag,
    g.phot_bp_mean_mag,
    g.phot_rp_mean_mag,
    g.phot_bp_mean_mag - g.phot_rp_mean_mag AS bp_rp

FROM gaiadr3.gaia_source AS g
JOIN TAP_UPLOAD.initial_cuts AS u
ON g.source_id = u.source_id

WHERE g.phot_g_mean_mag IS NOT NULL
  AND g.phot_bp_mean_mag IS NOT NULL
  AND g.phot_rp_mean_mag IS NOT NULL
  AND g.parallax IS NOT NULL

ORDER BY u.source_id
"""

job = Gaia.launch_job(   # <-- use sync for now (more stable)
    query=query,
    upload_resource="ruwe_cut_clean.vot",
    upload_table_name="initial_cuts"
)

results = job.get_results()

results.write("gaia_gbprp_ruwe_data.vot", format="csv", overwrite=True)

print(results[['SOURCE_ID','ra','dec','parallax','phot_g_mean_mag','phot_bp_mean_mag','phot_rp_mean_mag', 'bp_rp']])
print(results.colnames)